<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/Day_14_Mini_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import torch
import torch.nn as nn
import math

In [19]:
class MultiHeadAttention(nn.Module):

    def __init__(self, embed_dim, num_heads):
        super().__init__()

        assert embed_dim % num_heads == 0, \
            "Embedding dimension must be divisible by number of heads"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # Query, Key, Value projections
        self.Wq = nn.Linear(embed_dim, embed_dim)
        self.Wk = nn.Linear(embed_dim, embed_dim)
        self.Wv = nn.Linear(embed_dim, embed_dim)

        # Final output projection
        self.fc_out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):

        batch_size = x.shape[0]
        seq_len = x.shape[1]

        # Generate Q, K, V
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # Split into multiple heads
        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        # Move head dimension forward
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # Attention scores
        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        # Scale
        scores = scores / math.sqrt(self.head_dim)

        # Softmax
        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        # Context vectors
        context = torch.matmul(
            attention_weights,
            V
        )

        # Restore dimensions
        context = context.transpose(1, 2)

        context = context.reshape(
            batch_size,
            seq_len,
            self.embed_dim
        )

        # Final projection
        output = self.fc_out(context)

        return output

In [20]:
class Transformer(nn.Module):
  def __init__(self, embed_dim,num_heads):
      super().__init__()

      self.attention=MultiHeadAttention(
          embed_dim,
          num_heads
      )

      self.norm1=nn.LayerNorm(embed_dim)

      self.ffn=nn.Sequential(
          nn.Linear(embed_dim,embed_dim*4),
          nn.ReLU(),
          nn.Linear(embed_dim*4,embed_dim)
      )

      self.norm2=nn.LayerNorm(embed_dim)

  def forward(self,x):
    attention_output=self.attention(x)

    #residuaal + Norm

    x=self.norm1(x + attention_output)

    ffn_output=self.ffn(x)

    x=self.norm2(x+ffn_output)

    return x

In [21]:
#Mini GPT

In [22]:
class MiniGpt(nn.Module):
  def __init__(self,vocab_size,embed_dim,num_heads,num_layers):
     super().__init__()

     self.embedding=nn.Embedding(vocab_size,embed_dim)

     self.blocks=nn.Sequential(
         *[
             Transformer(embed_dim,num_heads)
             for _ in range(num_layers)
         ]
     )
     self.fc=nn.Linear(
         embed_dim,
         vocab_size
     )

  def forward(self,x):
    x=self.embedding(x)

    x=self.blocks(x)

    logits=self.fc(x)

    return logits



In [23]:
vocab = {
    "i":0,
    "love":1,
    "ai":2,
    "python":3,
    "data":4
}

In [24]:
tokens = torch.tensor([
    [0,1,2]
])

In [32]:
vocab_size = 5
model = MiniGpt(
    vocab_size=vocab_size,
    embed_dim=8,
    num_heads=2,
    num_layers=2
)

In [33]:
output = model(tokens)

print(output.shape)

torch.Size([1, 3, 5])


In [34]:
print(output)

tensor([[[-0.5222,  0.0738,  0.2266, -0.4216,  1.1027],
         [-0.3711, -0.2455, -0.8947,  0.6253,  0.3023],
         [-0.6359, -0.5651,  0.4790, -0.8198,  0.0970]]],
       grad_fn=<ViewBackward0>)


**What GPT Does Internally**

Input
↓
Predict next word
↓
Append
↓
Predict again
↓
Append
↓
Repeat

Exactly how ChatGPT, Claude, Gemini, Llama generate text.

### **Interview-Level Summary**

**Why Embedding?**
Converts token IDs into dense semantic vectors.

**Why Positional Encoding?**
Attention has no sense of order, positional information is added.

**Why Attention?**
Allows every token to look at every other token.

**Why Multi-Head Attention?**
Different heads learn different relationships.

**Why FFN?**
Processes information gathered by attention.

**Why Residual Connections?**
Prevent information loss and help gradient flow.

**Why LayerNorm?**
Stabilizes training.

**Why Linear + Softmax?**
Converts hidden representations into probabilities over the vocabulary.

**How GPT Generates Text?**
Predict one token → append → predict again → repeat.


In [35]:
tokens = torch.tensor([
    [0,1,2]
])

input_tokens = tokens[:,:-1]

target_tokens = tokens[:,1:]

print(input_tokens)
print(target_tokens)

tensor([[0, 1]])
tensor([[1, 2]])


In [36]:
loss_fn = nn.CrossEntropyLoss()

In [37]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [39]:
for epochs in range(100):
  logits=model(input_tokens)

  logits=logits.view(-1,vocab_size)

  targets=target_tokens.view(-1)
  loss = loss_fn(
        logits,
        targets
    )

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()

  print(f'At epoch {epochs} : loss : {loss.item()}')



At epoch 0 : loss : 0.33900201320648193
At epoch 1 : loss : 0.33550170063972473
At epoch 2 : loss : 0.33205273747444153
At epoch 3 : loss : 0.3286527991294861
At epoch 4 : loss : 0.3253006041049957
At epoch 5 : loss : 0.32199448347091675
At epoch 6 : loss : 0.3187330365180969
At epoch 7 : loss : 0.3155152201652527
At epoch 8 : loss : 0.3123399019241333
At epoch 9 : loss : 0.3092063069343567
At epoch 10 : loss : 0.3061133325099945
At epoch 11 : loss : 0.3030603528022766
At epoch 12 : loss : 0.30004680156707764
At epoch 13 : loss : 0.2970716953277588
At epoch 14 : loss : 0.2941344976425171
At epoch 15 : loss : 0.29123449325561523
At epoch 16 : loss : 0.28837111592292786
At epoch 17 : loss : 0.2855435907840729
At epoch 18 : loss : 0.2827513515949249
At epoch 19 : loss : 0.2799936532974243
At epoch 20 : loss : 0.27726995944976807
At epoch 21 : loss : 0.2745794355869293
At epoch 22 : loss : 0.2719215154647827
At epoch 23 : loss : 0.26929569244384766
At epoch 24 : loss : 0.266701340675354
At